# SAP RPT-1.5 / RPT-1.6 tabular prediction with direct REST calls

This notebook shows how to call the **SAP-RPT-1.5** and **SAP-RPT-1.6** tabular foundation models deployed on **SAP AI Core** with plain HTTP requests.

**Why direct REST?** The native SAP AI SDK (`gen_ai_hub.proxy.native.sap.RPTClient`, see `../RPT-1_native_sdk.ipynb`) models only the original RPT-1 request/response fields. RPT-1.5/1.6 added fields such as **explanations**, regression **confidence intervals** and **`context_mode`**, which older SDK models drop. Sending and receiving plain dictionaries keeps every field.

What you will learn:

1. The anatomy of one raw REST call (OAuth token, deployment URL, headers, body).
2. A joint **classification + regression** request with `top_k` candidates and confidence intervals.
3. **Explainability**: which columns and which context rows the model relied on.
4. How the four RPT-1.5/1.6 model variants compare on the same request.

**Prerequisites:** running AI Core deployments of `sap-rpt-1.5`, `sap-rpt-1.5-large`, `sap-rpt-1.6` and `sap-rpt-1.6-large` (scenario `foundation-models`, executable `aicore-sap`), and the `AICORE_*` variables from your service key in a `.env` file (see `.env.example`).

## 1. Setup

`rpt_client_api.py` (next to this notebook) is a small dependency-light client. It reads the `AICORE_*` variables, caches the OAuth token and finds the running deployment for a given model name, so no deployment ID has to be hard-coded.

In [1]:
import json
import time

import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv
from tqdm.auto import tqdm

from rpt_client_api import RPTClientAPI, flatten_predictions, map_explanation_rows

# Finds rest/.env or, walking up, RPT-1/python/.env.
load_dotenv(find_dotenv(usecwd=True))
pd.set_option("display.max_colwidth", 120)

## 2. Anatomy of a raw REST call

Before using the helper client, this is everything an RPT call needs, written with `requests` only:

| Step | Call |
|---|---|
| OAuth token | `POST {AICORE_AUTH_URL}/oauth/token` with `grant_type=client_credentials` |
| Deployment URL | `{AICORE_BASE_URL}/v2/inference/deployments/{deployment_id}` |
| Prediction | `POST {deployment_url}/predict` with headers `Authorization: Bearer <token>` and `AI-Resource-Group` |

The deployment URL is looked up by `RPTClientAPI` (it lists `foundation-models` configurations whose `modelName` matches and joins them with `RUNNING` deployments). We do not print it: deployment URLs and tokens should stay out of notebooks and logs.

In [2]:
import os

token = requests.post(
    os.environ["AICORE_AUTH_URL"].rstrip("/") + "/oauth/token",
    data={
        "grant_type": "client_credentials",
        "client_id": os.environ["AICORE_CLIENT_ID"],
        "client_secret": os.environ["AICORE_CLIENT_SECRET"],
    },
    timeout=60,
).json()["access_token"]

deployment_url = RPTClientAPI(model_name="sap-rpt-1.6").deployment_url

minimal_body = {
    "index_column": "ID",
    "prediction_config": {
        "target_columns": [
            {"name": "COSTCENTER", "prediction_placeholder": "[PREDICT]", "task_type": "classification"}
        ]
    },
    "rows": [
        {"ID": "1", "PRODUCT": "Office Chair", "COSTCENTER": "Office Furniture"},
        {"ID": "2", "PRODUCT": "Server Rack", "COSTCENTER": "Data Infrastructure"},
        {"ID": "3", "PRODUCT": "Standing Desk", "COSTCENTER": "[PREDICT]"},
    ],
}
raw = requests.post(
    deployment_url + "/predict",
    headers={
        "Authorization": f"Bearer {token}",
        "AI-Resource-Group": os.environ["AICORE_RESOURCE_GROUP"],
        "Content-Type": "application/json",
    },
    json=minimal_body,
    timeout=120,
)
print("HTTP", raw.status_code)
print(json.dumps({k: v for k, v in raw.json().items() if k != "id"}, indent=2))

HTTP 200
{
  "explanations": null,
  "metadata": {
    "context_mode": "default",
    "num_columns": 3,
    "num_predictions": 1,
    "num_query_rows": 1,
    "num_rows": 3
  },
  "predictions": [
    {
      "COSTCENTER": [
        {
          "confidence": 0.92,
          "confidence_interval": null,
          "prediction": "Data Infrastructure"
        }
      ],
      "ID": "3"
    }
  ],
  "status": {
    "code": 0,
    "message": "ok"
  }
}


With only two context rows the answer above is not meaningful (a desk is not data infrastructure); this cell only shows the call anatomy. Real use needs representative context rows, as in the next sections.

Two status layers exist and both must be checked:

* **HTTP status** (transport and schema validation; a malformed body returns 422).
* **`status.code`** inside the body: `0` ok, `1` warning, `2` invalid input, `3` server error.

`RPTClientAPI.predict_json` checks both and raises `RPTAPIError` with the server's message, so the rest of this notebook uses it.

## 3. Load the sample data

`../sample_orders.csv` is the small order table used by the other RPT language tracks. We hide `COSTCENTER` and `DISCOUNT_RATE` for two orders and let the model fill them in; the true values are kept aside to check the answers.

In [3]:
orders = pd.read_csv("../sample_orders.csv", dtype={"ID": str})
QUERY_IDS = ["35", "711"]
TARGETS = ["COSTCENTER", "DISCOUNT_RATE"]

truth = orders.loc[orders["ID"].isin(QUERY_IDS), ["ID", *TARGETS]].set_index("ID")
table = orders.copy()
# Targets become text columns so they can hold the "[PREDICT]" placeholder.
table[TARGETS] = table[TARGETS].astype(object)
table.loc[table["ID"].isin(QUERY_IDS), TARGETS] = "[PREDICT]"
table

,ID,PRODUCT,PRICE,ORDERDATE,QUANTITY,COSTCENTER,DISCOUNT_RATE
0,44,Office Chair,150.80,2025-11-02,12,Office Furniture,0.08
1,104,Server Rack,2200.00,2025-11-01,2,Data Infrastructure,0.15
2,88,Laptop Stand,49.90,2025-11-03,25,Office Furniture,0.05
3,205,Standing Desk,640.00,2025-11-05,6,Office Furniture,0.1
4,306,Monitor 27 inch,289.99,2025-11-08,10,Office Furniture,0.07
5,401,Network Switch,980.00,2025-11-10,3,Data Infrastructure,0.12
6,502,Keyboard Mechanical,120.00,2025-11-12,30,Office Furniture,0.06
7,35,Couch,999.99,2025-11-28,4,[PREDICT],[PREDICT]
8,610,Cable Management Kit,25.50,2025-11-15,50,Office Furniture,0.03
9,711,UPS Battery Backup,450.00,2025-11-18,5,[PREDICT],[PREDICT]


## 4. Build a joint multi-target request

Key `prediction_config` fields:

| Field | Meaning |
|---|---|
| `target_columns[].task_type` | `classification` or `regression` (set it explicitly instead of relying on auto-detection) |
| `target_columns[].top_k` | number of ranked candidates per classification cell |
| `explanations.top_column_scores` | per query row, the N most attended feature columns (1-20) |
| `explanations.top_relevant_context_rows` | per query row, positions of the N most relevant context rows (1-20) |
| `context_mode` | `default` or `deep` (RPT-1.6 only, see notebook 02) |

`data_schema` declares column types so `ORDERDATE` is read as a date and `PRICE` as a number.

> **Joint prediction:** RPT-1.6 predicts all target columns of a request together, so results can differ from two separate single-target requests.

In [4]:
payload = {
    "index_column": "ID",
    "prediction_config": {
        "target_columns": [
            {"name": "COSTCENTER", "prediction_placeholder": "[PREDICT]", "task_type": "classification", "top_k": 2},
            {"name": "DISCOUNT_RATE", "prediction_placeholder": "[PREDICT]", "task_type": "regression"},
        ],
        "explanations": {"top_column_scores": 3, "top_relevant_context_rows": 2},
    },
    "data_schema": {
        "ID": {"dtype": "string"},
        "PRODUCT": {"dtype": "string"},
        "PRICE": {"dtype": "numeric"},
        "ORDERDATE": {"dtype": "date"},
        "QUANTITY": {"dtype": "numeric"},
        "COSTCENTER": {"dtype": "string"},
        "DISCOUNT_RATE": {"dtype": "numeric"},
    },
    "rows": table.to_dict(orient="records"),
}

client = RPTClientAPI(model_name="sap-rpt-1.6")
response = client.predict_json(payload)
print("status:", response["status"])
print("metadata:", response["metadata"])

status: {'code': 0, 'message': 'ok'}
metadata: {'context_mode': 'default', 'num_columns': 7, 'num_predictions': 4, 'num_query_rows': 2, 'num_rows': 10}


## 5. Read the predictions

`flatten_predictions` turns the nested response (one record per query row, one candidate list per target) into long-form records:

* classification: one record per `top_k` rank with a `confidence` (probability);
* regression: one record with the point `prediction` and a `confidence_interval` `[lower, upper]` (`confidence` is `null`).

In [5]:
predictions = pd.DataFrame(flatten_predictions(response, index_column="ID"))
predictions["actual"] = [truth.loc[i, t] for i, t in zip(predictions["index"], predictions["target"])]
predictions

,query_position,index,target,rank,prediction,confidence,confidence_interval,actual
0,0,35,COSTCENTER,1,Office Furniture,0.86,None,Office Furniture
1,0,35,COSTCENTER,2,Data Infrastructure,0.14,None,Office Furniture
2,0,35,DISCOUNT_RATE,1,0.11661,NaN,"[0.0994, 0.1437]",0.09
3,1,711,COSTCENTER,1,Data Infrastructure,0.74,None,Data Infrastructure
4,1,711,COSTCENTER,2,Office Furniture,0.26,None,Data Infrastructure
5,1,711,DISCOUNT_RATE,1,0.092333,NaN,"[0.0505, 0.105]",0.11


## 6. Explainability

`map_explanation_rows` resolves the explanation row positions against the **complete submitted table** (positions are zero-based indexes into `rows`, not `ID` values).

How to read them:

* `column_scores` are attention-based importance scores of the query row. They describe what the model looked at, not causality. A truncated top-N list does not sum to 1, so do not renormalize it or show it as percentages.
* `relevant_context_rows` are the labeled examples the model found most relevant. They are not nearest neighbours and not proof of cause.
* In a joint request there is **one explanation per query row**, not per target.

In [6]:
for query_id, explanation in zip(QUERY_IDS, map_explanation_rows(response, payload["rows"])):
    print(f"Query ID {query_id}")
    print("  top columns:", explanation["column_scores"])
    for context in explanation["relevant_context_rows"]:
        row = context["row"]
        print(f"  context row {context['position']}: {row['PRODUCT']} -> {row['COSTCENTER']}, {row['DISCOUNT_RATE']}")

Query ID 35
  top columns: {'PRICE': 0.382, 'PRODUCT': 0.3, 'QUANTITY': 0.214}
  context row 5: Network Switch -> Data Infrastructure, 0.12
  context row 3: Standing Desk -> Office Furniture, 0.1
Query ID 711
  top columns: {'PRICE': 0.382, 'PRODUCT': 0.328, 'QUANTITY': 0.198}
  context row 5: Network Switch -> Data Infrastructure, 0.12
  context row 1: Server Rack -> Data Infrastructure, 0.15


## 7. Compare the RPT-1.5 and RPT-1.6 variants

Same payload, four deployments. `context_mode` is not set here because RPT-1.5 does not accept it. Expect small differences: the variants are different models, and with only eight context rows the answers rest on little evidence.

| Model | Max context rows | Recommended context | Max columns | Use |
|---|---|---|---|---|
| `sap-rpt-1.5` | 2,048 | 500-2,000 | 100 | compatibility, explainability |
| `sap-rpt-1.5-large` | 65,536 | 4,000-8,000 | 256 | compatibility, larger context |
| `sap-rpt-1.6` | 2,048 | 500-2,000 | 100 | default, low latency |
| `sap-rpt-1.6-large` | 65,536 | 4,000-36,000 | 256 | large context, high cardinality, `deep` mode |

All variants accept up to 512 query rows and 10 target columns per request (SAP Help, SAP-RPT-1.5 and SAP-RPT-1.6 model pages).

In [7]:
MODELS = ["sap-rpt-1.5", "sap-rpt-1.5-large", "sap-rpt-1.6", "sap-rpt-1.6-large"]
comparison = []
for model in tqdm(MODELS, desc="models"):
    model_client = RPTClientAPI(model_name=model)
    started = time.perf_counter()
    model_response = model_client.predict_json(payload)
    latency = time.perf_counter() - started
    for record in flatten_predictions(model_response, index_column="ID"):
        if record["rank"] == 1:
            comparison.append({"model": model, "latency_s": round(latency, 2), **record})

comparison = pd.DataFrame(comparison)
comparison.pivot_table(index=["index", "target"], columns="model", values="prediction", aggfunc="first")

models:   0%|          | 0/4 [00:00<?, ?it/s]

model                     sap-rpt-1.5 sap-rpt-1.5-large          sap-rpt-1.6  \
index target                                                                   
35    COSTCENTER     Office Furniture  Office Furniture     Office Furniture   
      DISCOUNT_RATE          0.120284          0.120284              0.11661   
711   COSTCENTER     Office Furniture  Office Furniture  Data Infrastructure   
      DISCOUNT_RATE          0.084428          0.084428             0.092333   

model                  sap-rpt-1.6-large  
index target                              
35    COSTCENTER        Office Furniture  
      DISCOUNT_RATE              0.11661  
711   COSTCENTER     Data Infrastructure  
      DISCOUNT_RATE             0.092333

In [8]:
comparison.groupby("model")["latency_s"].first().to_frame("request latency (s)")

,request latency (s)
model,
sap-rpt-1.5,0.53
sap-rpt-1.5-large,0.30
sap-rpt-1.6,0.23
sap-rpt-1.6-large,0.27


## 8. Larger payloads: gzip

For big tables, `predict_json(payload, compress=True)` gzips the JSON body (`Content-Encoding: gzip`). The response is identical.

In [9]:
compressed = client.predict_json(payload, compress=True)
print("same predictions:", compressed["predictions"] == response["predictions"])

same predictions: True


## Summary

* Call RPT on AI Core with `POST {deployment_url}/predict` and check both the HTTP status and `status.code`.
* Declare `task_type`, `top_k` and `data_schema` explicitly. Keep an `index_column` to join answers back.
* Regression answers include a `confidence_interval`; classification answers include ranked candidates with probabilities.
* Request `explanations` only when you use them, and resolve row positions against the full submitted table.
* Next: `02_RPT-1.6_deep_context.ipynb` for large context with `sap-rpt-1.6-large` and `context_mode="deep"`.